# RepairScope-X — Notebook 15
## ALoDLM-1.7B FP32 Model-Native Suffix Self-Repair

Notebook 14 successfully established the second genuine DLM runtime in **full FP32 precision** on a Tesla T4:

- **24/24** ALoDLM generations completed;
- peak VRAM ≈ **7.94 GB**;
- **16 genuine task failures** were collected;
- failures covered all **3 datasets**:
  - 7 Countdown
  - 5 GSM8K
  - 4 SVAMP
- `ALODLM_OWN_FAILURE_CORPUS_READY = true`.

This notebook asks the next model-generalization question:

> Can ALoDLM repair **its own failed outputs** when only a suffix is reopened, and how does the preservation/compute/success trade-off compare with a full regeneration under the same repair decoder?

## Important architectural distinction

Dream supports explicit masked-token reopening across a fixed response canvas.

ALoDLM's released portable decoder is different: it conditions on an **observed prefix** and generates future tokens with adaptive diffusion-style parallel commitment and recurrent refinement.

Therefore the faithful ALoDLM repair intervention is **suffix reopening**, not arbitrary interior remasking.

For an ALoDLM failure retokenized as `y = [y1 ... yL]`:

- `open_last_25`: preserve the first 75% of tokens as observed context; regenerate the last 25%.
- `open_last_50`: preserve the first 50%; regenerate the last 50%.
- `open_last_75`: preserve the first 25%; regenerate the last 75%.
- `global`: preserve none of the failed answer and regenerate the full answer budget.

## Why use the entropy repair decoder?

Notebook 14 collected failures with the quality-oriented deterministic `left1` decoder. Simply retrying the identical deterministic configuration would reproduce the same failure.

For the repair intervention, all four actions therefore use the **same official alternative ALoDLM decoder mode**:

- `mode="entropy"`
- `q=0.4`
- `tau=0.4`
- window size 16

This creates a meaningful same-model repair attempt while keeping the repair decoder fixed across scopes.

## Matrix

All **16 ALoDLM-origin failures** are used.

**16 failures × 4 repair scopes = 64 deterministic FP32 self-repair trials**

ALoDLM decoding is deterministic under this public decoder, so repeated “seeds” would duplicate identical outputs and are intentionally not used.

The notebook reports:

- task-aware correctness;
- exact preserved-prefix audit;
- scope-specific preservation;
- native adaptive compute:
  - outer steps,
  - recurrent passes,
  - loop-token passes,
  - mean exit depth,
  - wall time;
- same-wrong recurrence;
- per-failure oracle headroom;
- failure-cluster bootstrap confidence intervals;
- paired selective-vs-global exact tests.

Embedded Notebook-14 failure payload SHA-256:

`5051aea912767a41bcdfe0617dc0061a1497599de0907700cdca3489ad79fd06`

In [ ]:
!rm -rf /content/ALoDLM
!git clone -q https://github.com/amazon-science/ALoDLM.git /content/ALoDLM
!cd /content/ALoDLM && git checkout -q fe225a99e0b5aeff351804e871d7581155d7f27d
!pip -q install "transformers==4.56.1" "accelerate==1.10.1" "huggingface-hub>=0.34,<1.0" "safetensors>=0.6" "PyYAML>=6" scipy pandas
!pip -q install -e /content/ALoDLM --no-deps

## 1. Runtime preflight

In [ ]:
import os, gc, json, math, re, time, random, hashlib, platform, ast as pyast, subprocess, sys
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter

import numpy as np
import pandas as pd
import torch, transformers, accelerate, huggingface_hub
from scipy.stats import binomtest

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

BASE_SEED = 3700
random.seed(BASE_SEED)
np.random.seed(BASE_SEED)
torch.manual_seed(BASE_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(BASE_SEED)
    torch.cuda.empty_cache()

gc.collect()

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA:", torch.cuda.is_available())

assert torch.cuda.is_available(), "Notebook 15 requires a CUDA GPU runtime."

GPU_NAME = torch.cuda.get_device_name(0)
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30

print("GPU:", GPU_NAME, "| VRAM:", round(GPU_GB, 2), "GB")
print("Allocated:", round(torch.cuda.memory_allocated()/2**30, 3), "GB")

if GPU_GB < 14:
    raise RuntimeError("FP32 ALoDLM-1.7B expects roughly T4-class VRAM or larger.")

if torch.cuda.memory_allocated()/2**30 > 1.0:
    raise RuntimeError("GPU memory is already occupied. Restart runtime and run from the top.")

torch.set_float32_matmul_precision("high")
print("Runtime preflight: PASS")

## 2. Verify pinned ALoDLM source

In [ ]:
REPO_DIR = Path("/content/ALoDLM")
EXPECTED_CODE_COMMIT = "fe225a99e0b5aeff351804e871d7581155d7f27d"

actual_commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    text=True,
).strip()

assert actual_commit == EXPECTED_CODE_COMMIT

sys.path.insert(0, str(REPO_DIR))

from alodlm.generate import load_decoder
from alodlm.inference import DecodeConfig

print("ALoDLM commit:", actual_commit)
print("Pinned source validation: PASS")

## 3. Recover the exact 16 ALoDLM-origin failures

In [ ]:
import base64, zlib

EMBEDDED_PAYLOAD_SHA256 = "5051aea912767a41bcdfe0617dc0061a1497599de0907700cdca3489ad79fd06"

raw_payload = zlib.decompress(
    base64.b64decode('''eNrlnY1u40aSgF+F0AXIzMTW9v+PcbnF5JIgAWaTYDJBcBcHCkW1LGIkUuGPHWfWz3EPdC92VU1KIiVS5lhjZLxnIE5MdldXd7O/rqouMu9G8zBelpnLRxe/vBtducRlYRGnySSejS5GUVomxSy9SS6icBlPq1sX1BB5QfU/w2U6W64m8zVno7PROkunS7caUBEKz8IizF3RLDmZgxYLFBRmRYzl8e6udtXEao2VfspdUCxckJSrqcvygKqzgJKzIExmATOB+yOMiuVtkCYRlEsDqDcro6pKEWZXrgi4GAevXVFmSRAG19DKLAizuFisXBFHIGANyuTQqhfp4J8buBn89vW33718NXn53Y8/f/X6Ivj3qv3/+G0MylWCRxcgmZyN0rJYl6jqb7/9tr4tFmlymfxb8AYUONZYjF0JPoO+BOfQkctkncVJ8exy1Lo6en6ZgNhLEPnixXdp4S5evPCi6ykIMvd7GYPMoMzj5OoDjNQ36Y27dtmZv97Qt6lWAI2GyzwwZ0GSVtXeLKBDcTKLo7AAbcJgDcomRRwuA5dlaQajBlLjOd738/1ulBdZHBUTaAFqVReTcrk8G0EHoGjUulMNdXvgawF5GUWg5OhiDjq5XfXt9SIrHdbN3x6U9TM02T2Yuw5Xytx9kFaqh3lSpG9dAlctO9ssPzfbXVVno5twuZzkLkqTGVxRY6oNMVwqbuFHMZTk5nGzDBkLa4TkllhfRknQwUucrF1WlxtdUDmW3GhrGbECpApobOVC7HBcTGZuXSxGF2zMlZCGb39A1jJN15WKk3WY5wgPqjXruOHbq9ZCdR1KmnFLHmd+taBehVuDKKbPRpmLyiyDZ2XbAI5EXoD4zIW5Z8PSJVeFB4YL306us3A1uZqOLvTYKBRvqJLMoPSsTCZ5fJWEsNwdVAydm0Y0EkJG2s2FnXFNp1NBnNNQjUxtaKfW8VApTcOZiIjVbMpnU8epmEXUjFATEAZajdK38Ff1MNcPag3U9wXo3dl74ZeRB+KXkQ+PX5ixgKkaKnIIVJh5LPyC5KP4zdNlWXjWJtfp8noLybycFlnoweIbXZXLIl4vazRhN+DBixbtTlRqDRyK8UaFNvCfMY38VM+DF1jjc/iFBRvIhUKzFBRFrHrI+uF7OYc1A0qdOxi/ErRMrio+RymsnKjobkb6ZhQ2o6CZUwnMVJPA1dgPZWN9/bERfKyZBzOYGKK0JkYiX00PgomWnHGKRbSQqo/B2mhNpaRWChApOxkshLZUbX/6GGz1YAZrarTcJ6/8f0HePXa+L3nNQ8lrPjx5LVhzuqYNGQJeychjkRdFPwi9wzs0T7PVfm8G2tXPQNwLkP0cLFY73lnW7Y61Sm0M7VMhaZuMrEfpXw2Syu5DUowVlcYoywXBnx5DVYPlKXUNUst5NyXF2ArDNQfDl0uq+ijJhTZMEWhUAoJ4NyUFH2ipwkwBwRUg3BhBpKdri5fUdvFS831eujTvhCVRglkKY8OV+nhpaU6hJX8oLfkj0FIAXEQNFzuElueUPhYtUfQpcQIB1hzFX8zuYNa8eEqUoH+gxm2Sg6zreOZmIAMENW1oGFAwl2OwYfcH9FSc1iO3N5BP1u+nXTjV5BCnmhJBkKmISiE7cQpmpJLCMI9ToUwfTYE9mgqmgJcG7NRumvbgU4jB+FTMSGzH/+b71ibppKcYSE9pQXcMW1BFP1568pPoKR8IT/kIPj74teJ94oby0SKs8rQIKxNg3EEfgJI7cjYvnkLO/mEa7/v+jSYrfPr4QiWoXfNkL71lgIonbn92A1MeAtMyRjixxHZbnoYTYBm13jq1os89Z1wryqQiRIGZZntCpN2slIQMZiXZxyPtxKMealwyDhawpgw2jI+WjvIUOKoHwlE9wvkTOfMnK7jsKR1CRyrsY+ERRZ90AuVdYH+yQzeAbF475fTpyDjF2NUerbbRVz9sbxpBzl1MIb+oVPo6TsJl8DLJb1yGqgWgblP9k935tvlZj/YT5invDHqyfZ5KcIbBQddGK3+qJLqpqiQaldZTlTPei1WrwVTV1ggrpVHd/jynklHNLQVSg2uve6KenfTtjnrWkYFaqBx28jTUn9fEACUtbjkfL3LVKch9qDP/CL48+qfbg2w+CLmPBtyTcPvMO+yUPA/+Bh3pDUy2S32gyCTjLZT9KxqG4gBkfEyJIZaB33gsMGkVA8eVedZp2nOCzseSckqZoVRI4GM3xnY+MP7upJjRw71qUB5Mu/o33Y9Jsi6GYbhgEMO4oJJTLbkUhIiPFWL3O9VX+cq8vSiyME7A4xHsGLj2y7Zg5W8OQ9TXWZhEcf5pMA9XMbBoEebo2r0tl27p8rNABFdlDIstr6gVBpHLCmh04/Ol8+A6Tpdxko+Dr9Daqat6QSLAJZNcbe5VovwttXerErJf69t5BcO0APNo12J9PwhXaW2s7RRuqVur5mOPZFPtLFikN8EqTG639/1R9eFQXLu/N0/pW8h8c0SvOYzuViVs/PLyGYNfRbxy2L/PA3N5+bxS0Zetla6Lim1Rf+YtoOw4eAW7Ady7hj+CaXNbaU2Bd9d/L8PavMQaBj12dNvFNUgTxEv7MV1eo6E7T7ONVG+KXl7+cplwtisNd+En8X+Z+g/471MhblpH8E+b4ZTZoUfwRnHCNFiQyHDa7eMbAQI5Vb6MwPObbohbzIICsxBMUrAPu318agcdwb+HMUqaWVAHaVCcnngYbxlY1lILpRVsUvovZflRHN/Lb8n0YH5D2Yfy++dwiSk1c7dcVijx2Y7x1QLswPUyLPzpM6IpXbkgj8L5PF3OcOV79mAtmKUi+DK8RnvPp+fAU77EEiIA0w/MVo9hkFq1AcBAl3c284oB/opF5lywSjOAkodWsQiLwD+CAKo5Xl/ERbFxq68yWJ6zin3490ZJoDmUdNcuAfTm6zByswCeLXSxF74/Z8ENCt526ga2ikrtNGliWh1geuPkb4/wN4NWd7R33FBHfIix0PER+sYTOanGqGt8DobmGWV17ef9w1QB/cgQQYEUBIKYVZy4SslP820Nz3TwIV68eOO3qlkMDz96G9gDl7x4cbHtQ/Bs9yBgJsFWveBZozNY7zlsBVRtun6ZMJT/w2bMUC/oBIqut6xgETQ2rWoy4a/ixsGAAc6AaWURXzf6+WGDx+qJby+ED91eBAEDmAnjfQRGeo7bYP9hilSOBNO2N3nBEsUkAWlcct2dZiuUT8jq2lQMEcPzupqbFB4AfuAML4OHbopZ8pdneB3dIu7dU7QevqdA2Qf7BMB4XKq1HZy5MoESZ8EPuOLRqpyWt8gpgMAaKZXjBjNNE3cOsI4WcQTz6tm5CJfzinlgqcOq8TAPV9MSFiZSE7AJsLpKPfRSFAu+h99JdnIXKRAzvfLs7KoNayZti8gxBxVYlcezDXJvYA0EN0iejU5uo1KjCRS58Qx2CszTdFYJgEogv7nX8HbUusXineOyJ6saRi8RxMG25gKYiKgE+rnqPC0u3Aq6dlOhEpdBML31/94B/Yv2eCNvt/OxgfI329HC299gz9Oq83WtM7ALPKIvL+dZGL2j6u4du6tN/ec7adxLq4cJZbFNp6p9vz01W6ne9cDdgrTFCRT3I07PqaBvv0xhn7ojQYaSXuLbD8rznhDeDXrFtGRW2iqVDYPj3aDXQhsljdKMGExK6PIj6KDXKagZiHxgsm0KPPAj+Kl+BNEwOpZz2MWge/Ljgf4+w++DvqbWDoU+lj0F+gjO22Aav3XA1iRDlC6XGC9xSP11CAbwJ6a55HfXMXwNKhSeLijBew2VFFicefCJPCxTR2JKQJ6PwKDdCzbjRmBeGdOzupbxdfK/t44NRT+At7IbWhwX7Vk8jzOwg3dExlcZXJglVWQHR8m/UgHreRxsafwDisE9CKjsZXngNnu/YfKPrvId2gW7x3ddS/VAhQH8DG9/HnwC3j+2/J37o9jbPnabTkvlun+we5w3la0vowog/X//J/DSBTnZ/hakfdz41NM35DAu6zGTUnFVxXd6gvSSS8sYFcyXEX3hHcYpsdQKoTkY4T0pHLr3xbbhEZ2DA0V7qrmtBR6CKmrEX32meBSk95KXUzqYvFD2oeR9GeQlzMzCZSms5ATzquorJZii4FA755kF44iWFNjkGFRO0HOPE5i5Ouzrq1zDwxbWcXSo+xX8ff6PdApV6thDFmMABHy/Wg7UAVCW2Xhn+lY35qA4qOQ1KjbivY6oR5X4gBUrUhX7CmA131gT15IcxGa2YsHerXpaRa23FimajgWA7l2l193dO9G44vt/hxYrG8t2Sd+1qsRd4IPe3+6UfqbIpvbzs8P+VSrUIqtIPEE7Vu5rUwk+GP56mFEIDvWhXqhEo/Isns8drLfIbbpP5ckpH7KVQUeeehREDrWNFSMUGMxEdQzaYxuDsU2EUhWEe3OOJaGMEsW1ZgCzHgq3f3qRzIe/a6w63nM7+Q1j2HUkuANggH5MUZADynZhOb8OV+u6QgQe/DknRyMhXeVbePYFhuEZ1ifYYxh5BRZtFmoRTNP0be5NSNW4ukqv44rOyJRPgygL/wQzGp4++B0t0nQJF3NYBa4Kq9+mJWaJzfANiNo1r0Wg5JuwiBaARGo3N6tWd6hGU7G65jFc18WTy63oFoB7Y+M4dT5a8vDOHvZ111Vs6vfS5dWryjkIWwzpwxRDxNiJvc85bLT2wZ8Vfp4hTepQDwxkYwRh5CrR++LQfgZvIccDAf+6sjekk3qrxah0PHNZ5+nzbjDqhiqJb3qL+gnGJEW7O/bYL9kQBUVPZn/79b0nbn2boeTnnEkqrKnSo3mP+W2U4pR58lPN+nL9qLQGTHUtDa/Y3RX+xgh5D+6NHG6BW36Ie3WyFQ6+A/oZVv3VydT30nsQ8oV6L+IL9VDg/zf49yE47jNwx7kM1mW+OC/X1YrnQPQyASh77F3droJoCXMDy3kW3o7rQ0KsKGs/flt7WhaB0QEYgPlOhgfen1V74z2sb2v6QtsqKHyrYdoKRrN7YiH3iW50vDMSsj1dbI7IpvY4GDZuRwfJ61PL8UEPDL7sBGWukaSC4RCJAQvL8dKveGcnunuo90Vv77ZFB+eB+aAZLvypp7gQOjQCQqzSgDMlPV97QiASacK4T1Q01PTmKTLNBZNCGiOgbCeDAeRiQIqLJINxzFjzOHL/NJKTk8EspSGYki6NwvzPj4nMQr0/mK0U70VmKP9QNDMLOFqtMSvhBi3QLL3Z5LIo1b61iN/irdo4BcxgNHSMn5HpklAX8+kdSZom+zA+UqcWXQFmX06Lz73G9/btkzl+eAxkLT5I45VNehVfu2Sbgo6mfWMUO6WdNQezcxzrFHbVI6ehwQl29rbbi/RY11GTzUvce2mR90jYG6i37vbDflqIP3G7Ww/OO6GcKqpFZVOLHubjNzCEElVuuu5NO1GWUSmUskQT0fOOTevwkD9GEPx081sxI6S1HLYn3Lo+JsrvE3sY5hV/P8wr/lDMV58Jc/77YEEYLBxUARY6n03nbqu17E/xqhOvL8oIIbUIV8E6XIb1e8uZOwteg8HnlmAKhlmCyMEIgZDBdZzHRQpkAEL4V286pQRpUkcb3HWclmB7o43/7TwwQNC1S9dLV0lysx4B+DpODRufXWckymgEcLaKHJfjFcHwfdhKP9FGDN9T9jPG64YPO//Dts0QQ/IR4hBbrneSHaq3MlKA5rZvmwhHR1ud41mj16e35JscTZjaKheyXGPMfpvSvns9c0+FTauVjs0WPs23pbYHtpvMxD+CnnT6tsLNUajOcd/cMwhxXh8qrO6Tt0uEbeh8svPRiv3Xj8kTdj8oGZ4CqYxVpgrvYF5711ZkqZDE0xncj74PjcJWZLiRzCohtSbdCZC9yY9m+Oud/FhGPRMnZ8IQQ2DThY33r8+EuXdTufsVnpS0cBg4pWJyBfOND/8qnbklqroK/0yTv718lX756h/ndKy/gOb8zQmunuo5HIG7GGnQzArluCN2PhNWzUIyhZEw0xlTc6FCLVHxCOrC47xaxcjPuWNMhtY6MpWhm8+5xDRWZzSdQXFKpZzpOdMzv1fBEq7bq7fD3Q5az1uZxL+XbtLaWetb/qnffcaaqu1OmU9w05g0C4AluRWGl2AJ7LbeuvYqTuJVuZq036ujuxsd79eBHqvwj8nBEwPuuzCKa26Jfwfj5avvYbgn3//83eTrl9+++un1V5P//P71Dz/9OHn91csv/6v6ctBda+Jgc4vnLi9w8iJ4SHHZlkWELhxh6pyScyreUH3BtF9oGBDgnxFyQfBLr4/wgMJ4FWmULieA1XrWqseO6uk5gOx8Mxfn13iO72AnmM1wnMPbZRrOJvkiZFJBLQHCpzzkU9Bhbg2jU2cdo85GcxpSo9TMzBX8WzKplWNyziPJprPQgFpuGoanPHPVg+4X0gdfCB3P8+/+lVJ8Vsv6v7ABz5p5gaMEvhRsAjBVf7r6IYSnKXE3LUC3ntRf7v162j3fwLjvU5X3fF/ovm8M3/M6+H1fiD/WORywjly95iUFO83epTqNuyPVpC2L75eq3wnteM2o1aLlaKT32O49p6iHQSMqOi+rzqvadrYoO2VQ0imEwhDATnG1LtHqdfkyDN5gfbgw8WaZRxme3ytOYXSMqj8dC1ZYtEAKjfGLGJ9FJeU47yA6yfHVE6ADLnKoN6ajNtLAlluF2a3/fw4c+y5Aaxsw+/bQJPNbms/G2DOJtrf8aVDTLtreoUwe7B2bD6F1fF9IVaL87b3v9DTudHywxzvEXY63IZqDYSUJJ9Sgp+8LdX5yzdrxton9r3KAJQL8pMR/691sco67D+KgNaskN8bndXRvWe3PWKA7e5iJ9ZjzsruzmxfZOy9Gj/25Yce8YAZh77SYvmlhVgvoO44n0+LYrCiuqoeoY1rUmDHN/Wf4fBxHHZkWcCioIrC4wK5kWjE+yJRoT8wmIHD6xPRNy+GkiGOLpW9SaGPIur5utXvO94+HhVSUMHguGWVKHVstmvXOixxzJcEOEIpAETyJOTItwDOtrJYaFg3DnP/OadnzCe5+vfs/qPQJGw==''')
)

assert hashlib.sha256(raw_payload).hexdigest() == EMBEDDED_PAYLOAD_SHA256

UPSTREAM = json.loads(raw_payload.decode("utf-8"))
FAILURES = UPSTREAM["failures"]
UPSTREAM_GATE = UPSTREAM["notebook14_gate"]
UPSTREAM_MANIFEST = UPSTREAM["notebook14_manifest"]

assert len(FAILURES) == 16
assert len({x["failure_id"] for x in FAILURES}) == 16
assert len({x["problem_id"] for x in FAILURES}) == 16
assert Counter(x["dataset"] for x in FAILURES) == {
    "countdown_fresh": 7,
    "gsm8k": 5,
    "svamp": 4,
}
assert all(not x["task_success"] for x in FAILURES)
assert UPSTREAM_GATE["ALODLM_OWN_FAILURE_CORPUS_READY"] is True
assert UPSTREAM_MANIFEST["precision"] == "fp32"

MODEL_ID = UPSTREAM_MANIFEST["model_id"]
MODEL_REVISION = UPSTREAM_MANIFEST["model_revision"]

print("Payload SHA:", EMBEDDED_PAYLOAD_SHA256)
print("Failures:", len(FAILURES))
print("Distribution:", dict(Counter(x["dataset"] for x in FAILURES)))
print("Frozen model revision:", MODEL_REVISION)
print("Failure corpus validation: PASS")

## 4. Task-aware verifier

In [ ]:
FINAL_RE = re.compile(
    r"(?im)^[ \t]*FINAL_ANSWER[ \t]*:[ \t]*"
    r"([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)"
    r"[ \t]*$"
)
NUM_RE = re.compile(r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?")

def close_num(a, b):
    return a is not None and math.isclose(float(a), float(b), abs_tol=1e-8, rel_tol=1e-8)

def strict_final(text):
    vals = FINAL_RE.findall(str(text))
    if not vals:
        return None
    try:
        x = float(vals[-1])
        return x if math.isfinite(x) else None
    except Exception:
        return None

def last_number(text):
    vals = NUM_RE.findall(str(text).replace(",", ""))
    if not vals:
        return None
    try:
        x = float(vals[-1])
        return x if math.isfinite(x) else None
    except Exception:
        return None

COUNTDOWN_PROMPT_RE = re.compile(
    r"Use the numbers\s+(.+?)\s+exactly once\s+to produce the target\s+"
    r"([-+]?(?:\d+(?:\.\d*)?|\.\d+))",
    flags=re.I,
)
ARITH_RUN_RE = re.compile(r"[0-9eE+\-*/().=\s]{3,}")
ALLOWED_BINOPS = (pyast.Add, pyast.Sub, pyast.Mult, pyast.Div)
ALLOWED_UNARY = (pyast.UAdd, pyast.USub)

def parse_countdown_prompt(prompt):
    m = COUNTDOWN_PROMPT_RE.search(str(prompt))
    if m is None:
        raise ValueError("Countdown prompt format not recognized.")
    return [float(x) for x in NUM_RE.findall(m.group(1))], float(m.group(2))

def safe_arith(expr):
    try:
        tree = pyast.parse(expr, mode="eval")
    except Exception:
        return None

    literals = []

    def walk(node):
        if isinstance(node, pyast.Expression):
            return walk(node.body)

        if isinstance(node, pyast.Constant) and isinstance(node.value, (int, float)):
            v = float(node.value)
            if not math.isfinite(v):
                raise ValueError
            literals.append(v)
            return v

        if isinstance(node, pyast.BinOp) and isinstance(node.op, ALLOWED_BINOPS):
            a, b = walk(node.left), walk(node.right)
            if isinstance(node.op, pyast.Add): return a + b
            if isinstance(node.op, pyast.Sub): return a - b
            if isinstance(node.op, pyast.Mult): return a * b
            if isinstance(node.op, pyast.Div):
                if abs(b) < 1e-15: raise ZeroDivisionError
                return a / b

        if isinstance(node, pyast.UnaryOp) and isinstance(node.op, ALLOWED_UNARY):
            v = walk(node.operand)
            return -v if isinstance(node.op, pyast.USub) else v

        raise ValueError

    try:
        value = walk(tree)
        return (float(value), literals) if math.isfinite(value) else None
    except Exception:
        return None

def same_multiset(a, b):
    if len(a) != len(b):
        return False
    aa, bb = sorted(map(float, a)), sorted(map(float, b))
    return all(close_num(x, y) for x, y in zip(aa, bb))

def countdown_task_success(text, prompt):
    nums, target = parse_countdown_prompt(prompt)

    cleaned = (
        str(text)
        .replace("×", "*")
        .replace("÷", "/")
        .replace("\\times", "*")
        .replace("\\div", "/")
    )

    candidates = []

    for match in ARITH_RUN_RE.finditer(cleaned):
        run = match.group(0).strip()
        if any(op in run for op in ["+", "-", "*", "/"]):
            candidates.extend(
                part.strip()
                for part in run.split("=")
                if part.strip()
            )

    for expr in sorted(set(candidates), key=len, reverse=True):
        parsed = safe_arith(expr)

        if parsed is None:
            continue

        value, literals = parsed

        if same_multiset(literals, nums) and close_num(value, target):
            return True, expr

    return False, None

def verify_repair(text, failure):
    strict = strict_final(text)
    numeric = last_number(text)

    strict_ok = close_num(strict, failure["target"])
    numeric_ok = close_num(numeric, failure["target"])

    if failure["dataset"] == "countdown_fresh":
        task_ok, expr = countdown_task_success(
            text,
            failure["prompt"],
        )
    else:
        task_ok, expr = numeric_ok, None

    return {
        "strict_prediction": strict,
        "numeric_prediction": numeric,
        "target": float(failure["target"]),
        "strict_success": bool(strict_ok),
        "numeric_success": bool(numeric_ok),
        "task_success": bool(task_ok),
        "valid_countdown_expression": expr,
    }

demo = "Use the numbers 19, 29, and 24 exactly once to produce the target 14."
assert countdown_task_success("19 + 24 - 29 = 14", demo)[0]
assert not countdown_task_success("29 - 19 = 10", demo)[0]

for failure in FAILURES:
    assert not verify_repair(failure["output"], failure)["task_success"]

print("Task verifier + starting-failure audit: PASS")

## 5. Load the exact Notebook-14 model snapshot in FP32

In [ ]:
from huggingface_hub import snapshot_download
from types import SimpleNamespace

MODEL_DIR = snapshot_download(
    repo_id=MODEL_ID,
    revision=MODEL_REVISION,
)

torch.cuda.empty_cache()
gc.collect()
torch.cuda.reset_peak_memory_stats()

args = SimpleNamespace(
    model=MODEL_DIR,
    model_config=None,
    device="cuda",
    precision="fp32",
)

t0 = time.time()

decoder = load_decoder(args)

load_seconds = time.time() - t0

tokenizer = decoder.tokenizer
model = decoder.model

assert next(model.parameters()).dtype == torch.float32

print("MODEL LOAD: PASS")
print("Model:", MODEL_ID)
print("Revision:", MODEL_REVISION)
print("Precision: FP32")
print("Load seconds:", round(load_seconds, 2))
print("Allocated GB:", round(torch.cuda.memory_allocated()/2**30, 3))

## 6. Frozen repair-decoder configuration

In [ ]:
REPAIR_Q = 0.4
REPAIR_TAU = 0.4
REPAIR_MODE = "entropy"
REPAIR_WINDOW = 16

ACTIONS = [
    "open_last_25",
    "open_last_50",
    "open_last_75",
    "global",
]

print({
    "q": REPAIR_Q,
    "tau": REPAIR_TAU,
    "mode": REPAIR_MODE,
    "window_size": REPAIR_WINDOW,
    "actions": ACTIONS,
})

## 7. Prompt and suffix-repair construction

In [ ]:
PROMPT_SUFFIX = (
    "\n\nSolve the problem carefully. "
    "For ordinary arithmetic word problems, end with `FINAL_ANSWER: <number>`. "
    "For Countdown, include a valid arithmetic expression using every supplied number exactly once."
)

def base_prompt_ids(failure):
    ids = tokenizer.apply_chat_template(
        [{"role": "user", "content": failure["prompt"] + PROMPT_SUFFIX}],
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    return list(ids)

def response_ids(text):
    return tokenizer(
        text,
        add_special_tokens=False,
        return_attention_mask=False,
    )["input_ids"]

def scope_definition(length, action):
    if action == "global":
        return 0, length

    preserve_fraction = {
        "open_last_25": 0.75,
        "open_last_50": 0.50,
        "open_last_75": 0.25,
    }[action]

    cut = max(
        1,
        min(
            length - 1,
            int(math.floor(length * preserve_fraction)),
        ),
    )

    return cut, length - cut

for failure in FAILURES:
    ids = response_ids(failure["output"])

    if len(ids) < 4:
        raise RuntimeError(
            "A starting failure is too short for suffix intervention: "
            + failure["failure_id"]
        )

    for action in ACTIONS:
        cut, budget = scope_definition(len(ids), action)
        assert budget >= 1
        if action == "global":
            assert cut == 0
        else:
            assert cut >= 1

print("Suffix-scope construction: PASS")

## 8. ALoDLM suffix self-repair runner

For selective repair:

1. retokenize the exact failed ALoDLM output;
2. retain the chosen prefix tokens;
3. append those tokens to the observed chat-generation prompt;
4. generate only the reopened suffix budget with the official ALoDLM decoder;
5. reconstruct the full answer from the retained prefix + newly generated suffix.

The retained token prefix is therefore observed context and cannot change.

`global` uses no failed-response prefix and generates a full response-length budget using the exact same entropy repair decoder.

In [ ]:
@torch.inference_mode()
def run_repair(failure, action):
    original_ids = response_ids(
        failure["output"]
    )

    response_len = len(
        original_ids
    )

    cut, suffix_budget = scope_definition(
        response_len,
        action,
    )

    preserved_ids = (
        original_ids[:cut]
        if cut > 0
        else []
    )

    prompt_ids = (
        base_prompt_ids(
            failure
        )
        + preserved_ids
    )

    config = DecodeConfig(
        q=REPAIR_Q,
        tau=REPAIR_TAU,
        mode=REPAIR_MODE,
        window_size=REPAIR_WINDOW,
        max_new_tokens=suffix_budget,
        position_penalty=None,
    )

    config.validate()

    torch.cuda.reset_peak_memory_stats()

    result = decoder.generate(
        prompt_ids,
        config,
    )

    suffix_ids = list(
        result[
            "token_ids"
        ]
    )

    repaired_ids = (
        preserved_ids
        + suffix_ids
    )

    # Mechanical preservation is true by construction, but audit explicitly.
    assert repaired_ids[:cut] == preserved_ids

    repaired_text = tokenizer.decode(
        repaired_ids,
        skip_special_tokens=True,
    ).strip()

    verification = verify_repair(
        repaired_text,
        failure,
    )

    original_numeric = last_number(
        failure["output"]
    )

    repaired_numeric = verification[
        "numeric_prediction"
    ]

    same_wrong_numeric = bool(
        not verification[
            "task_success"
        ]
        and original_numeric is not None
        and repaired_numeric is not None
        and close_num(
            original_numeric,
            repaired_numeric,
        )
    )

    exact_token_position_preservation = (
        sum(
            int(a == b)
            for a, b in zip(
                original_ids,
                repaired_ids,
            )
        )
        / max(
            len(original_ids),
            len(repaired_ids),
            1,
        )
    )

    forced_fraction = (
        cut
        / response_len
    )

    return {
        "trial_id":
            failure["failure_id"]
            + "|"
            + action,
        "failure_id":
            failure["failure_id"],
        "problem_id":
            failure["problem_id"],
        "dataset":
            failure["dataset"],
        "action":
            action,
        "original_response_tokens":
            response_len,
        "preserved_prefix_tokens":
            cut,
        "requested_suffix_budget":
            suffix_budget,
        "generated_suffix_tokens":
            int(
                result[
                    "generated_tokens"
                ]
            ),
        "forced_fraction":
            float(
                forced_fraction
            ),
        "strict_success":
            bool(
                verification[
                    "strict_success"
                ]
            ),
        "numeric_success":
            bool(
                verification[
                    "numeric_success"
                ]
            ),
        "task_success":
            bool(
                verification[
                    "task_success"
                ]
            ),
        "same_wrong_numeric":
            same_wrong_numeric,
        "verification":
            verification,
        "original_output":
            failure["output"],
        "repaired_output":
            repaired_text,
        "exact_prefix_preserved":
            True,
        "token_position_preservation":
            float(
                exact_token_position_preservation
            ),
        "outer_steps":
            int(
                result[
                    "outer_steps"
                ]
            ),
        "recurrent_passes":
            int(
                result[
                    "recurrent_passes"
                ]
            ),
        "mean_exit_depth":
            result[
                "mean_exit_depth"
            ],
        "loop_token_passes":
            int(
                result[
                    "loop_token_passes"
                ]
            ),
        "loop_token_passes_per_output_token":
            result[
                "loop_token_passes_per_output_token"
            ],
        "wall_seconds":
            float(
                result[
                    "wall_seconds"
                ]
            ),
        "tokens_per_second":
            float(
                result[
                    "tokens_per_second"
                ]
            ),
        "stop_reason":
            str(
                result[
                    "stop_reason"
                ]
            ),
        "peak_vram_gb":
            float(
                torch.cuda.max_memory_allocated()
                / 2**30
            ),
    }

## 9. One real FP32 suffix-repair smoke test

In [ ]:
smoke = run_repair(
    FAILURES[0],
    "open_last_75",
)

print(
    json.dumps(
        {
            k: v
            for k, v in smoke.items()
            if k not in {
                "original_output",
                "repaired_output",
                "verification",
            }
        },
        indent=2,
    )
)

print("\nORIGINAL FAILURE\n")
print(smoke["original_output"])

print("\nALoDLM SUFFIX REPAIR\n")
print(smoke["repaired_output"])

assert smoke["exact_prefix_preserved"]
assert smoke["outer_steps"] > 0
assert smoke["recurrent_passes"] > 0

print("\nREAL ALoDLM FP32 SUFFIX SELF-REPAIR: PASS")

## 10. Output directory + run signature

In [ ]:
SAVE_TO_DRIVE = True
CLEAR_OLD_RESULTS = False

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = Path(
        "/content/drive/MyDrive/RepairScopeX/Notebook15_ALoDLM_FP32_SelfRepair"
    )
else:
    OUT_DIR = Path(
        "/content/RepairScopeX_Notebook15_ALoDLM_FP32_SelfRepair"
    )

OUT_DIR.mkdir(parents=True, exist_ok=True)

signature_payload = {
    "protocol_version":
        "alodlm17b-fp32-suffix-selfrepair-v1",
    "embedded_payload_sha256":
        EMBEDDED_PAYLOAD_SHA256,
    "code_commit":
        EXPECTED_CODE_COMMIT,
    "model_id":
        MODEL_ID,
    "model_revision":
        MODEL_REVISION,
    "precision":
        "fp32",
    "repair_q":
        REPAIR_Q,
    "repair_tau":
        REPAIR_TAU,
    "repair_mode":
        REPAIR_MODE,
    "repair_window":
        REPAIR_WINDOW,
    "actions":
        ACTIONS,
    "failure_ids":
        [
            x[
                "failure_id"
            ]
            for x in FAILURES
        ],
}

RUN_SIGNATURE = hashlib.sha256(
    json.dumps(
        signature_payload,
        sort_keys=True,
    ).encode(
        "utf-8"
    )
).hexdigest()

TRIALS_PATH = (
    OUT_DIR
    / "alodlm_self_repair_trials.jsonl"
)

MANIFEST_PATH = (
    OUT_DIR
    / "run_manifest.json"
)

if CLEAR_OLD_RESULTS:
    for p in [
        TRIALS_PATH,
        MANIFEST_PATH,
    ]:
        if p.exists():
            p.unlink()

if MANIFEST_PATH.exists():
    old = json.loads(
        MANIFEST_PATH.read_text(
            encoding="utf-8"
        )
    )

    if old.get(
        "run_signature"
    ) != RUN_SIGNATURE:
        raise RuntimeError(
            "Existing Notebook 15 outputs use a different configuration."
        )

else:
    MANIFEST_PATH.write_text(
        json.dumps(
            {
                "created_utc":
                    datetime.now(
                        timezone.utc
                    ).isoformat(),
                "run_signature":
                    RUN_SIGNATURE,
                **signature_payload,
                "gpu":
                    GPU_NAME,
                "gpu_total_gb":
                    GPU_GB,
                "torch":
                    torch.__version__,
                "transformers":
                    transformers.__version__,
            },
            indent=2,
        ),
        encoding="utf-8",
    )

print("Run signature:", RUN_SIGNATURE)
print("Output:", OUT_DIR)

## 11. Resume-safe 64-trial self-repair sweep

In [ ]:
def load_completed(path):
    done = {}

    if not path.exists():
        return done

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as f:

        for line_number, line in enumerate(
            f,
            start=1,
        ):
            if not line.strip():
                continue

            try:
                rec = json.loads(
                    line
                )
            except json.JSONDecodeError as exc:
                raise RuntimeError(
                    f"Malformed JSONL at line {line_number}."
                ) from exc

            if rec.get(
                "run_signature"
            ) != RUN_SIGNATURE:
                raise RuntimeError(
                    "Found trial from another Notebook 15 configuration."
                )

            tid = rec[
                "trial_id"
            ]

            if tid in done:
                raise RuntimeError(
                    f"Duplicate trial_id: {tid}"
                )

            done[
                tid
            ] = rec

    return done


EXPECTED_IDS = {
    failure[
        "failure_id"
    ]
    + "|"
    + action
    for failure in FAILURES
    for action in ACTIONS
}

assert len(
    EXPECTED_IDS
) == 64

completed = load_completed(
    TRIALS_PATH
)

print(
    "Already completed:",
    len(
        completed
    ),
    "/ 64",
)

counter = 0

with open(
    TRIALS_PATH,
    "a",
    encoding="utf-8",
) as f:

    for failure in FAILURES:
        for action in ACTIONS:
            counter += 1

            tid = (
                failure[
                    "failure_id"
                ]
                + "|"
                + action
            )

            if tid in completed:
                print(
                    f"[{counter}/64] SKIP {tid}"
                )
                continue

            try:
                rec = run_repair(
                    failure,
                    action,
                )

                rec[
                    "run_signature"
                ] = RUN_SIGNATURE
                rec[
                    "status"
                ] = "ok"
                rec[
                    "error"
                ] = None

            except torch.cuda.OutOfMemoryError as exc:
                torch.cuda.empty_cache()
                gc.collect()

                rec = {
                    "trial_id":
                        tid,
                    "failure_id":
                        failure[
                            "failure_id"
                        ],
                    "problem_id":
                        failure[
                            "problem_id"
                        ],
                    "dataset":
                        failure[
                            "dataset"
                        ],
                    "action":
                        action,
                    "status":
                        "oom",
                    "error":
                        str(
                            exc
                        ),
                    "run_signature":
                        RUN_SIGNATURE,
                }

            except Exception as exc:
                rec = {
                    "trial_id":
                        tid,
                    "failure_id":
                        failure[
                            "failure_id"
                        ],
                    "problem_id":
                        failure[
                            "problem_id"
                        ],
                    "dataset":
                        failure[
                            "dataset"
                        ],
                    "action":
                        action,
                    "status":
                        "error",
                    "error":
                        f"{type(exc).__name__}: {exc}",
                    "run_signature":
                        RUN_SIGNATURE,
                }

            f.write(
                json.dumps(
                    rec,
                    ensure_ascii=False,
                )
            )
            f.write(
                "\n"
            )
            f.flush()

            if rec[
                "status"
            ] == "ok":
                print(
                    f"[{counter}/64] {tid} | "
                    f"task={rec['task_success']} | "
                    f"samewrong={rec['same_wrong_numeric']} | "
                    f"passes={rec['recurrent_passes']} | "
                    f"loop={rec['loop_token_passes']}"
                )
            else:
                print(
                    f"[{counter}/64] {tid} | "
                    f"{rec['status']} | {rec['error']}"
                )

print("ALoDLM self-repair sweep finished.")

## 12. Validate exactly 64 trials

In [ ]:
records = load_completed(
    TRIALS_PATH
)

missing = (
    EXPECTED_IDS
    - set(
        records
    )
)

extra = (
    set(
        records
    )
    - EXPECTED_IDS
)

print("Expected:", len(EXPECTED_IDS))
print("Found:", len(records))
print("Missing:", len(missing))
print("Extra:", len(extra))

if missing:
    raise RuntimeError(
        "Self-repair sweep incomplete. Rerun the sweep cell."
    )

if extra:
    raise RuntimeError(
        "Unexpected Notebook 15 trial IDs exist."
    )

rows = [
    records[
        tid
    ]
    for tid in sorted(
        EXPECTED_IDS
    )
]

status_counts = Counter(
    row[
        "status"
    ]
    for row in rows
)

print(
    "Statuses:",
    dict(
        status_counts
    ),
)

if status_counts.get(
    "oom",
    0,
):
    raise RuntimeError(
        "At least one FP32 ALoDLM repair OOMed."
    )

if status_counts.get(
    "error",
    0,
):
    raise RuntimeError(
        "At least one ALoDLM repair errored."
    )

TRIAL_DF = pd.DataFrame(
    rows
)

assert len(
    TRIAL_DF
) == 64

assert TRIAL_DF[
    "trial_id"
].nunique() == 64

assert TRIAL_DF[
    "exact_prefix_preserved"
].all()

assert (
    TRIAL_DF[
        "outer_steps"
    ] > 0
).all()

assert (
    TRIAL_DF[
        "recurrent_passes"
    ] > 0
).all()

print(
    "64-trial ALoDLM self-repair matrix integrity: PASS"
)

## 13. Action and dataset summaries

In [ ]:
action_rows = []

for action, group in TRIAL_DF.groupby(
    "action"
):
    failed = group[
        ~group[
            "task_success"
        ]
    ]

    action_rows.append({
        "action":
            action,
        "trials":
            len(
                group
            ),
        "task_success_rate":
            float(
                group[
                    "task_success"
                ].mean()
            ),
        "strict_success_rate":
            float(
                group[
                    "strict_success"
                ].mean()
            ),
        "numeric_success_rate":
            float(
                group[
                    "numeric_success"
                ].mean()
            ),
        "mean_forced_fraction":
            float(
                group[
                    "forced_fraction"
                ].mean()
            ),
        "mean_token_position_preservation":
            float(
                group[
                    "token_position_preservation"
                ].mean()
            ),
        "same_wrong_given_failure":
            float(
                failed[
                    "same_wrong_numeric"
                ].mean()
            )
            if len(
                failed
            )
            else 0.0,
        "mean_outer_steps":
            float(
                group[
                    "outer_steps"
                ].mean()
            ),
        "mean_recurrent_passes":
            float(
                group[
                    "recurrent_passes"
                ].mean()
            ),
        "mean_loop_token_passes":
            float(
                group[
                    "loop_token_passes"
                ].mean()
            ),
        "mean_exit_depth":
            float(
                group[
                    "mean_exit_depth"
                ].dropna().mean()
            )
            if group[
                "mean_exit_depth"
            ].notna().any()
            else math.nan,
        "mean_wall_seconds":
            float(
                group[
                    "wall_seconds"
                ].mean()
            ),
        "max_peak_vram_gb":
            float(
                group[
                    "peak_vram_gb"
                ].max()
            ),
    })

ACTION_DF = pd.DataFrame(
    action_rows
)

DATASET_DF = (
    TRIAL_DF.groupby(
        [
            "dataset",
            "action",
        ],
        as_index=False,
    )
    .agg(
        trials=(
            "task_success",
            "size",
        ),
        task_success_rate=(
            "task_success",
            "mean",
        ),
        same_wrong_rate=(
            "same_wrong_numeric",
            "mean",
        ),
        mean_preservation=(
            "token_position_preservation",
            "mean",
        ),
        mean_loop_token_passes=(
            "loop_token_passes",
            "mean",
        ),
        mean_wall_seconds=(
            "wall_seconds",
            "mean",
        ),
    )
)

display(
    ACTION_DF.sort_values(
        [
            "task_success_rate",
            "mean_loop_token_passes",
        ],
        ascending=[
            False,
            True,
        ],
    )
)

display(
    DATASET_DF
)

## 14. Per-failure repair surface + deterministic oracle headroom

In [ ]:
SURFACE_DF = (
    TRIAL_DF[
        [
            "failure_id",
            "problem_id",
            "dataset",
            "action",
            "task_success",
            "same_wrong_numeric",
            "forced_fraction",
            "token_position_preservation",
            "outer_steps",
            "recurrent_passes",
            "loop_token_passes",
            "mean_exit_depth",
            "wall_seconds",
        ]
    ]
    .copy()
)

fixed = (
    SURFACE_DF.groupby(
        "action",
        as_index=False,
    )
    .agg(
        mean_task_success=(
            "task_success",
            "mean",
        ),
        mean_loop_token_passes=(
            "loop_token_passes",
            "mean",
        ),
        mean_preservation=(
            "token_position_preservation",
            "mean",
        ),
    )
    .sort_values(
        [
            "mean_task_success",
            "mean_loop_token_passes",
            "mean_preservation",
        ],
        ascending=[
            False,
            True,
            False,
        ],
    )
)

best_fixed = fixed.iloc[0]

oracle_rows = []

for failure_id, group in SURFACE_DF.groupby(
    "failure_id"
):
    best = group.sort_values(
        [
            "task_success",
            "loop_token_passes",
            "token_position_preservation",
        ],
        ascending=[
            False,
            True,
            False,
        ],
    ).iloc[0]

    oracle_rows.append({
        "failure_id":
            failure_id,
        "problem_id":
            best[
                "problem_id"
            ],
        "dataset":
            best[
                "dataset"
            ],
        "oracle_action":
            best[
                "action"
            ],
        "oracle_task_success":
            float(
                best[
                    "task_success"
                ]
            ),
        "oracle_loop_token_passes":
            float(
                best[
                    "loop_token_passes"
                ]
            ),
        "oracle_preservation":
            float(
                best[
                    "token_position_preservation"
                ]
            ),
    })

ORACLE_DF = pd.DataFrame(
    oracle_rows
)

best_fixed_success = float(
    best_fixed[
        "mean_task_success"
    ]
)

oracle_success = float(
    ORACLE_DF[
        "oracle_task_success"
    ].mean()
)

ORH = float(
    oracle_success
    - best_fixed_success
)

oracle_action_distribution = (
    ORACLE_DF[
        "oracle_action"
    ]
    .value_counts(
        normalize=True
    )
    .to_dict()
)

print(
    "Best fixed:",
    best_fixed[
        "action"
    ],
)

print(
    "Best fixed success:",
    round(
        best_fixed_success,
        4,
    ),
)

print(
    "Oracle success:",
    round(
        oracle_success,
        4,
    ),
)

print(
    "ALoDLM ORH:",
    round(
        ORH,
        4,
    ),
)

print(
    "Oracle action distribution:",
    oracle_action_distribution,
)

display(
    ORACLE_DF
)

## 15. Failure-cluster bootstrap confidence intervals

Because ALoDLM decoding is deterministic, the **16 failure instances** are the sampling units.

Bootstrap confidence intervals therefore resample failure IDs, not duplicate decoding seeds.

In [ ]:
def bootstrap_action_rate(
    action,
    n_boot=20000,
    seed=15001,
):
    sub = TRIAL_DF[
        TRIAL_DF[
            "action"
        ] == action
    ].copy()

    fids = sorted(
        sub[
            "failure_id"
        ].unique()
    )

    values = {
        fid: float(
            sub.loc[
                sub[
                    "failure_id"
                ] == fid,
                "task_success",
            ].iloc[0]
        )
        for fid in fids
    }

    observed = float(
        np.mean(
            list(
                values.values()
            )
        )
    )

    rng = np.random.default_rng(
        seed
    )

    boots = np.empty(
        n_boot,
        dtype=float,
    )

    for i in range(
        n_boot
    ):
        sampled = rng.choice(
            fids,
            size=len(
                fids
            ),
            replace=True,
        )

        boots[
            i
        ] = float(
            np.mean(
                [
                    values[
                        fid
                    ]
                    for fid in sampled
                ]
            )
        )

    lo, hi = np.quantile(
        boots,
        [
            0.025,
            0.975,
        ],
    )

    return (
        observed,
        float(
            lo
        ),
        float(
            hi
        ),
    )


ci_rows = []

for idx, action in enumerate(
    ACTIONS
):
    est, lo, hi = bootstrap_action_rate(
        action,
        seed=15001 + idx,
    )

    ci_rows.append({
        "action":
            action,
        "task_success":
            est,
        "ci95_low":
            lo,
        "ci95_high":
            hi,
    })

ACTION_CI_DF = pd.DataFrame(
    ci_rows
)

display(
    ACTION_CI_DF
)

## 16. Paired selective-vs-global success tests

Each selective result is paired with the `global` result for the **same failed response**.

The test is an exact discordant-pair test. Both one-sided and two-sided values are reported; no superiority claim is forced.

In [ ]:
contrast_rows = []

global_rows = (
    TRIAL_DF[
        TRIAL_DF[
            "action"
        ] == "global"
    ][
        [
            "failure_id",
            "task_success",
        ]
    ]
    .rename(
        columns={
            "task_success":
                "global_success"
        }
    )
)

for action in [
    "open_last_25",
    "open_last_50",
    "open_last_75",
]:
    selective_rows = (
        TRIAL_DF[
            TRIAL_DF[
                "action"
            ] == action
        ][
            [
                "failure_id",
                "task_success",
            ]
        ]
        .rename(
            columns={
                "task_success":
                    "selective_success"
            }
        )
    )

    paired = selective_rows.merge(
        global_rows,
        on="failure_id",
        how="inner",
    )

    assert len(
        paired
    ) == 16

    s = paired[
        "selective_success"
    ].astype(int).to_numpy()

    g = paired[
        "global_success"
    ].astype(int).to_numpy()

    selective_only = int(
        np.sum(
            (s == 1)
            & (g == 0)
        )
    )

    global_only = int(
        np.sum(
            (s == 0)
            & (g == 1)
        )
    )

    discordant = (
        selective_only
        + global_only
    )

    if discordant:
        p_one = float(
            binomtest(
                selective_only,
                discordant,
                p=0.5,
                alternative="greater",
            ).pvalue
        )

        p_two = float(
            binomtest(
                selective_only,
                discordant,
                p=0.5,
                alternative="two-sided",
            ).pvalue
        )

    else:
        p_one = 1.0
        p_two = 1.0

    contrast_rows.append({
        "selective_action":
            action,
        "selective_success_rate":
            float(
                s.mean()
            ),
        "global_success_rate":
            float(
                g.mean()
            ),
        "difference_vs_global":
            float(
                s.mean()
                - g.mean()
            ),
        "selective_only":
            selective_only,
        "global_only":
            global_only,
        "one_sided_p_selective_gt_global":
            p_one,
        "two_sided_p":
            p_two,
    })

CONTRAST_DF = pd.DataFrame(
    contrast_rows
)

display(
    CONTRAST_DF
)

## 17. Native repair-energy / efficiency frontier

For ALoDLM, a useful model-native compute proxy is `loop_token_passes`.

The table below reports success, preservation and compute jointly. It does not pretend that suffix and global actions have identical computational cost; the cost difference is itself part of RepairScope-X.

In [ ]:
ENERGY_DF = ACTION_DF[
    [
        "action",
        "task_success_rate",
        "mean_token_position_preservation",
        "same_wrong_given_failure",
        "mean_outer_steps",
        "mean_recurrent_passes",
        "mean_loop_token_passes",
        "mean_exit_depth",
        "mean_wall_seconds",
    ]
].copy()

global_compute = float(
    ENERGY_DF.loc[
        ENERGY_DF[
            "action"
        ] == "global",
        "mean_loop_token_passes",
    ].iloc[0]
)

ENERGY_DF[
    "loop_pass_fraction_vs_global"
] = (
    ENERGY_DF[
        "mean_loop_token_passes"
    ]
    / global_compute
)

display(
    ENERGY_DF
)

## 18. Second-DLM self-repair gate

Progression passes when:

1. all 64 FP32 trials complete;
2. every selective prefix is mechanically preserved;
3. at least 4 repairs are task-successful overall;
4. successes occur on at least 2 datasets;
5. at least 2 actions have non-zero success;
6. at least 2 successes come from selective suffix repair.

This gate tests whether ALoDLM provides a useful second-model repair signal. It does **not** require suffix repair to beat global regeneration.

In [ ]:
successful_repairs = int(
    TRIAL_DF[
        "task_success"
    ].sum()
)

datasets_with_success = int(
    TRIAL_DF.loc[
        TRIAL_DF[
            "task_success"
        ],
        "dataset",
    ].nunique()
)

actions_with_success = int(
    TRIAL_DF.loc[
        TRIAL_DF[
            "task_success"
        ],
        "action",
    ].nunique()
)

selective = TRIAL_DF[
    TRIAL_DF[
        "action"
    ] != "global"
]

selective_successes = int(
    selective[
        "task_success"
    ].sum()
)

GATE = {
    "model":
        MODEL_ID,
    "model_revision":
        MODEL_REVISION,
    "precision":
        "fp32",
    "repair_mode":
        REPAIR_MODE,
    "trials":
        int(
            len(
                TRIAL_DF
            )
        ),
    "unique_trial_ids":
        int(
            TRIAL_DF[
                "trial_id"
            ].nunique()
        ),
    "task_successful_repairs":
        successful_repairs,
    "selective_task_successful_repairs":
        selective_successes,
    "datasets_with_task_success":
        datasets_with_success,
    "actions_with_task_success":
        actions_with_success,
    "best_fixed_action":
        str(
            best_fixed[
                "action"
            ]
        ),
    "best_fixed_task_success":
        best_fixed_success,
    "oracle_task_success":
        oracle_success,
    "oracle_repair_headroom":
        ORH,
    "oracle_actions_used":
        int(
            ORACLE_DF[
                "oracle_action"
            ].nunique()
        ),
    "all_prefix_audits_pass":
        bool(
            TRIAL_DF[
                "exact_prefix_preserved"
            ].all()
        ),
    "max_peak_vram_gb":
        float(
            TRIAL_DF[
                "peak_vram_gb"
            ].max()
        ),
}

GATE[
    "ALODLM_SELF_REPAIR_SIGNAL_READY"
] = bool(
    GATE[
        "trials"
    ] == 64
    and GATE[
        "unique_trial_ids"
    ] == 64
    and GATE[
        "all_prefix_audits_pass"
    ]
    and GATE[
        "task_successful_repairs"
    ] >= 4
    and GATE[
        "datasets_with_task_success"
    ] >= 2
    and GATE[
        "actions_with_task_success"
    ] >= 2
    and GATE[
        "selective_task_successful_repairs"
    ] >= 2
)

print(
    json.dumps(
        GATE,
        indent=2,
    )
)

## 19. Save, hash, ZIP and download

In [ ]:
ACTION_PATH = (
    OUT_DIR
    / "alodlm_self_repair_action_summary.csv"
)

DATASET_PATH = (
    OUT_DIR
    / "alodlm_self_repair_dataset_summary.csv"
)

SURFACE_PATH = (
    OUT_DIR
    / "alodlm_self_repair_surface.csv"
)

ORACLE_PATH = (
    OUT_DIR
    / "alodlm_self_repair_oracle.csv"
)

CI_PATH = (
    OUT_DIR
    / "alodlm_self_repair_action_ci.csv"
)

CONTRAST_PATH = (
    OUT_DIR
    / "alodlm_selective_vs_global.csv"
)

ENERGY_PATH = (
    OUT_DIR
    / "alodlm_repair_energy.csv"
)

GATE_PATH = (
    OUT_DIR
    / "ALODLM_SELF_REPAIR_GATE.json"
)

ACTION_DF.to_csv(
    ACTION_PATH,
    index=False,
)

DATASET_DF.to_csv(
    DATASET_PATH,
    index=False,
)

SURFACE_DF.to_csv(
    SURFACE_PATH,
    index=False,
)

ORACLE_DF.to_csv(
    ORACLE_PATH,
    index=False,
)

ACTION_CI_DF.to_csv(
    CI_PATH,
    index=False,
)

CONTRAST_DF.to_csv(
    CONTRAST_PATH,
    index=False,
)

ENERGY_DF.to_csv(
    ENERGY_PATH,
    index=False,
)

GATE_PATH.write_text(
    json.dumps(
        GATE,
        indent=2,
    ),
    encoding="utf-8",
)

def sha256_file(path):
    h = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as f:
        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b"",
        ):
            h.update(
                chunk
            )

    return h.hexdigest()

artifact_paths = [
    MANIFEST_PATH,
    TRIALS_PATH,
    ACTION_PATH,
    DATASET_PATH,
    SURFACE_PATH,
    ORACLE_PATH,
    CI_PATH,
    CONTRAST_PATH,
    ENERGY_PATH,
    GATE_PATH,
]

ARTIFACT_MANIFEST = {
    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
    "run_signature":
        RUN_SIGNATURE,
    "embedded_payload_sha256":
        EMBEDDED_PAYLOAD_SHA256,
    "protocol_version":
        "alodlm17b-fp32-suffix-selfrepair-v1",
    "files": {
        p.name:
            sha256_file(
                p
            )
        for p in artifact_paths
        if p.exists()
    },
}

ARTIFACT_MANIFEST_PATH = (
    OUT_DIR
    / "artifact_manifest.json"
)

ARTIFACT_MANIFEST_PATH.write_text(
    json.dumps(
        ARTIFACT_MANIFEST,
        indent=2,
    ),
    encoding="utf-8",
)

import shutil

ZIP_BASE = Path(
    "/content/RepairScopeX_Notebook15_ALoDLM_FP32_SelfRepair_results"
)

zip_path = shutil.make_archive(
    str(
        ZIP_BASE
    ),
    "zip",
    root_dir=
        OUT_DIR,
)

print(
    "Created:",
    zip_path,
)

print(
    "Size MB:",
    round(
        Path(
            zip_path
        ).stat().st_size
        / 1024**2,
        3,
    ),
)

from google.colab import files

files.download(
    zip_path
)

# Progression rule

Send back:

`RepairScopeX_Notebook15_ALoDLM_FP32_SelfRepair_results.zip`

If `ALODLM_SELF_REPAIR_SIGNAL_READY = true`, RepairScope-X will have:

- independently confirmed Dream self-repair;
- a second genuine DLM;
- a **full-precision** ALoDLM self-repair result;
- model-native adaptive-compute measurements.

The next priority should then be a **cross-model synthesis / robustness notebook**, not another isolated pilot: harmonize Dream and ALoDLM repair surfaces, preservation, recurrence and compute into the main paper tables and identify the minimum remaining experiments before manuscript freeze.